# Chapter 60: Causal Bandits and Thompson Sampling

## Learning Objectives
- Implement Thompson sampling for Bernoulli bandits
- Compare with epsilon-greedy exploration
- Evaluate contextual policy value with IPS
- Visualize reward trajectories and policy quality

## Prerequisites
- Chapters 0-55 completed
- Strong understanding of ML systems and evaluation
- Optional matplotlib for richer visual outputs


# Chapter 60: Causal Bandits and Thompson Sampling

## Why this chapter matters
Product decision systems need both exploration and causal reasoning. Thompson Sampling is a practical Bayesian bandit strategy, and causal estimators help evaluate policies under logged propensities.

## Learning goals
1. Implement Bernoulli Thompson Sampling.
2. Compare with epsilon-greedy under non-stationary/heterogeneous segments.
3. Estimate policy value with inverse propensity scoring (IPS).
4. Visualize cumulative reward and regret.

## Thompson Sampling (Beta-Bernoulli)
For each arm `a`:
- posterior \(\theta_a \sim Beta(\alpha_a, \beta_a)\)
- sample \(\tilde{p}_a\) and choose arm with max sample
- update posterior from observed reward

## Causal evaluation with logged data
IPS estimate:
\[
\hat{V}_{IPS}=\frac{1}{N}\sum_{i=1}^{N}\frac{\mathbb{1}(a_i=\pi(x_i))r_i}{p(a_i|x_i)}
\]

## Assignment
1. Run contextual segments with different arm effectiveness.
2. Evaluate policy with IPS and compare to online reward.
3. Analyze exploration-exploitation balance over time.


In [ ]:
from __future__ import annotations

import random
from typing import Dict, List, Tuple


def segment_reward_prob(segment: int, arm: int) -> float:
    # contextual/causal heterogeneity by segment
    table = {
        0: {0: 0.08, 1: 0.12, 2: 0.10},
        1: {0: 0.14, 1: 0.10, 2: 0.16},
        2: {0: 0.11, 1: 0.18, 2: 0.09},
    }
    return table[segment][arm]


def sample_segment() -> int:
    r = random.random()
    if r < 0.35:
        return 0
    if r < 0.75:
        return 1
    return 2


def run_thompson(T: int = 12000, n_arms: int = 3, seed: int = 42):
    random.seed(seed)
    alpha = [1.0] * n_arms
    beta = [1.0] * n_arms

    cum_reward = 0
    rewards = []
    log = []  # (segment, chosen_arm, propensity, reward)

    for _t in range(T):
        seg = sample_segment()
        samples = [random.betavariate(alpha[a], beta[a]) for a in range(n_arms)]
        arm = max(range(n_arms), key=lambda a: samples[a])

        # Thompson propensity approximated via Monte Carlo for IPS logging.
        mc = 120
        wins = 0
        for _ in range(mc):
            ss = [random.betavariate(alpha[a], beta[a]) for a in range(n_arms)]
            if arm == max(range(n_arms), key=lambda a: ss[a]):
                wins += 1
        prop = max(1e-3, wins / mc)

        p = segment_reward_prob(seg, arm)
        r = 1 if random.random() < p else 0
        cum_reward += r

        alpha[arm] += r
        beta[arm] += (1 - r)

        rewards.append(cum_reward)
        log.append((seg, arm, prop, r))

    return rewards, log


def run_epsilon_greedy(T: int = 12000, n_arms: int = 3, eps: float = 0.1, seed: int = 42):
    random.seed(seed)
    counts = [0] * n_arms
    values = [0.0] * n_arms

    cum_reward = 0
    rewards = []

    for _t in range(T):
        seg = sample_segment()
        if random.random() < eps:
            arm = random.randrange(n_arms)
        else:
            arm = max(range(n_arms), key=lambda a: values[a])

        p = segment_reward_prob(seg, arm)
        r = 1 if random.random() < p else 0
        cum_reward += r

        counts[arm] += 1
        values[arm] += (r - values[arm]) / counts[arm]

        rewards.append(cum_reward)

    return rewards


def ips_eval(log: List[Tuple[int, int, float, int]], policy) -> float:
    s = 0.0
    n = len(log)
    for seg, arm, prop, r in log:
        a_new = policy(seg)
        if a_new == arm:
            s += r / prop
    return s / n


def best_arm_policy(seg: int) -> int:
    # oracle per segment (for evaluation reference)
    best = {
        0: 1,
        1: 2,
        2: 1,
    }
    return best[seg]


def global_best_policy(_seg: int) -> int:
    return 1


def maybe_plot(ts_rewards: List[int], eg_rewards: List[int]):
    try:
        import matplotlib.pyplot as plt
    except Exception:
        print("Final reward Thompson:", ts_rewards[-1])
        print("Final reward EpsGreedy:", eg_rewards[-1])
        return

    plt.figure(figsize=(8, 4))
    plt.plot(ts_rewards, label="Thompson")
    plt.plot(eg_rewards, label="Epsilon-Greedy")
    plt.xlabel("Round")
    plt.ylabel("Cumulative Reward")
    plt.title("Bandit Policy Comparison")
    plt.legend()
    plt.tight_layout()
    plt.show()


if __name__ == "__main__":
    ts_rewards, ts_log = run_thompson(T=9000, n_arms=3, seed=7)
    eg_rewards = run_epsilon_greedy(T=9000, n_arms=3, eps=0.12, seed=7)

    print("Final cumulative reward Thompson:", ts_rewards[-1])
    print("Final cumulative reward EpsGreedy:", eg_rewards[-1])

    ips_oracle = ips_eval(ts_log, best_arm_policy)
    ips_global = ips_eval(ts_log, global_best_policy)

    print("IPS value (oracle contextual policy):", round(ips_oracle, 5))
    print("IPS value (global arm policy)      :", round(ips_global, 5))

    maybe_plot(ts_rewards, eg_rewards)


## Checkpoint

1. You can update Beta posteriors from rewards.
2. You can explain exploration via posterior uncertainty.
3. You can compute IPS policy value from logged bandit data.

## Guided Exercise
Change segment probabilities and compare policy robustness.

In [ ]:
# TODO (Student Exercise)
# Modify segment_reward_prob table and compare final rewards and IPS estimates.

## Quick Quiz

**Q1.** Why Thompson sampling explores naturally?

**Answer:** Posterior sampling gives uncertain arms chance proportional to uncertainty.

**Q2.** What does IPS correct for?

**Answer:** Biased logged action probabilities in offline policy evaluation.

**Q3.** Why contextual policies can outperform global arm policies?

**Answer:** Optimal action differs by user/context segment.


## Homework
Add doubly-robust policy evaluation and compare with IPS variance.